# Markov regression: final results

Run the six code stages from top to bottom. The notebook fits each model once;
regression tables, state probabilities and Wald tests share those fitted objects.
The last cell displays the final report. Detailed coefficients are expandable.

BH uses three separate families under `ALL`: 30 levels, 120 individual slopes
and 30 joint slopes, following `main_J1108.tex` and `Supplementary_Appendix.tex`.
Stage 5 adds the supplementary probability-weighted sensitivity checks.

This notebook is self-contained: model estimation, report formatting,
supplementary diagnostics and optional TeX verification are defined here.
No project-local Python module or Python plotting script is required.
Set VERIFY_MANUSCRIPT and the manuscript paths in Stage 6 to run the optional
numerical comparison after the final workbook has been created.


## 1. Paths, dependencies and shared utilities

This cell includes the report and optional verification helpers. Expand it to inspect them.


In [1]:
# Project setup: paths and shared model utilities; run before analysis.
from pathlib import Path
import hashlib
import warnings
import json
import math
import re
from html import escape
from openpyxl.styles import Alignment, Font, PatternFill

import numpy as np
import pandas as pd
from scipy.stats import norm
import statsmodels.api as sm
from IPython.display import HTML, display


def find_project_root():
    """Locate the parent of the Code and Data package folders."""
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / "my paper"):
            if ((candidate / "Energy Economics Code" / "matlab code").is_dir()
                and (candidate / "Energy Economics Data").is_dir()):
                return candidate.resolve()
    raise FileNotFoundError("Project root not found. Start the notebook with the kernel working "
        "directory inside the project folder containing Energy Economics Code "
        "and Energy Economics Data. See readme.txt.")

PROJECT_ROOT = find_project_root()
PACKAGE_ROOT = PROJECT_ROOT / "Energy Economics Code"
DATA_ROOT = PROJECT_ROOT / "Energy Economics Data"
RESILIENCE_FILE = PACKAGE_ROOT / "matlab code" / "resilience_metrics.xlsx"
MACRO_DATA_FILE = DATA_ROOT / "markov_scale_data_166.csv"
OUTPUT_ROOT = PACKAGE_ROOT / "python code" / "output"
NOTEBOOK_OUTPUT_DIR = OUTPUT_ROOT
OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR

for label, input_file in (("Resilience workbook", RESILIENCE_FILE),
    ("166-observation conditioning data", MACRO_DATA_FILE),):
    if not input_file.is_file():
        raise FileNotFoundError(f"{label} not found: {input_file}")

input_sha256 = {str(p.relative_to(PROJECT_ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
                for p in (RESILIENCE_FILE, MACRO_DATA_FILE)}

# Shared model utilities used by the single estimation pass.


def get_parameter_names(result):
    """Return parameter names from a statsmodels result."""
    if hasattr(result.params, "index"):
        return list(result.params.index)

    return list(result.model.param_names)


def make_named_series(values, names):
    """Convert a parameter array into a named pandas Series."""
    return pd.Series(np.asarray(values, dtype=float), index=names)


def find_parameter(names, target):
    """Find a parameter using exact string matching."""
    for name in names:
        if str(name) == target:
            return name

    return None


def find_regime_parameter(names, variable, regime, exog_columns=None):
    """
    Find a regime-dependent parameter.

    This supports both possible statsmodels naming formats:
        DXY[0], GSCPI[1], ...
    and:
        x1[0], x2[1], ...
    """
    if variable == "const":
        candidates = [f"const[{regime}]"]
    else:
        candidates = [f"{variable}[{regime}]"]

        if exog_columns is not None and variable in exog_columns:
            variable_position = exog_columns.index(variable) + 1
            candidates.append(f"x{variable_position}[{regime}]")

    for candidate in candidates:
        parameter_name = find_parameter(names, candidate)

        if parameter_name is not None:
            return parameter_name

    raise ValueError(f"Cannot find a regime parameter for variable '{variable}', "
        f"regime {regime}.\n"
        f"Tried parameter names: {candidates}\n"
        f"Available parameters:\n{names}")


def significance_stars(p_value):
    """Return conventional significance stars."""
    if pd.isna(p_value):
        return ""

    if p_value < 0.01:
        return "***"

    if p_value < 0.05:
        return "**"

    if p_value < 0.10:
        return "*"

    return ""


def calculate_z_p(estimate, standard_error):
    """Calculate a z-statistic and its two-sided normal p-value."""
    if (pd.isna(estimate)
        or pd.isna(standard_error)
        or standard_error == 0):
        return np.nan, np.nan

    z_value = estimate / standard_error
    p_value = 2.0 * norm.sf(abs(z_value))

    return z_value, p_value


def clean_small_number(value, decimals=4):
    """Avoid displaying rounded values such as -0.0000."""
    if pd.isna(value):
        return value

    threshold = 0.5 * 10 ** (-decimals)

    if abs(value) < threshold:
        return 0.0

    return value


def paper_format(estimate, z_value, p_value):
    """
    Produce the format used in the LaTeX table:

        0.8251*** (313.762)
       -0.0031    (-1.247)
    """
    estimate = clean_small_number(estimate, decimals=4)
    z_value = clean_small_number(z_value, decimals=3)
    stars = significance_stars(p_value)

    return f"{estimate:.4f}{stars} ({z_value:.3f})"


def validate_uncertainty_selection(selection):
    """Validate and normalize the uncertainty-index selection."""
    normalized = str(selection).strip().upper()
    valid_options = set(uncertainty_variable_maps) | {"ALL"}

    if normalized not in valid_options:
        raise ValueError(f"SELECTED_UNCERTAINTY must be one of "
            f"{sorted(valid_options)}; received '{selection}'.")

    if normalized == "ALL":
        return list(uncertainty_variable_maps.keys())

    return [normalized]


def validate_variable_codes(r_data, uncertainty_name, variable_map):
    """Check that all required Var codes are present in the Excel file."""
    available_codes = set(r_data["Variable"].astype(str))
    missing_codes = [code
        for code in variable_map.values()
        if code not in available_codes]

    if missing_codes:
        raise ValueError(f"The following variable codes required for {uncertainty_name} "
            f"are missing from resilience_metrics.xlsx: {missing_codes}")


def build_dependent_dataframe(r_data, value_column, variable_map):
    """
    Build a five-market dependent-variable DataFrame.

    Parameters
    ----------
    value_column : str
        RP -> resilience intensity (RI)
        RD -> resilience duration (RD)
    variable_map : dict
        Market-to-Var-code mapping for GPR, CPU, or OPU.
    """
    market_series = {}

    for market in markets:
        variable_code = variable_map[market]

        market_series[market] = (pd.to_numeric(r_data.loc[
                    r_data["Variable"].astype(str) == variable_code, value_column,],
                errors="coerce")
            .reset_index(drop=True))

    return pd.DataFrame(market_series)


def relabel_regimes(measure_name, regime_results):
    """
    Relabel raw statsmodels regimes according to economic meaning.

    RI:
        State 1 = high absorption = larger conditional mean.
        State 2 = low absorption  = smaller conditional mean.

    RD:
        State 1 = fast recovery = smaller conditional mean.
        State 2 = slow recovery = larger conditional mean.
    """
    if measure_name == "RI":
        ordered_raw_regimes = sorted([0, 1],
            key=lambda regime: regime_results[regime]["conditional_mean"], reverse=True)
    elif measure_name == "RD":
        ordered_raw_regimes = sorted([0, 1],
            key=lambda regime: regime_results[regime]["conditional_mean"])
    else:
        raise ValueError(f"Unknown resilience measure: {measure_name}")

    return ordered_raw_regimes[0], ordered_raw_regimes[1]


def paper_display_columns():
    """Return a fresh list of the existing compact-table columns."""
    return ["Market",
        "State 1 Cond. mean", "State 1 DXY", "State 1 GSCPI", "State 1 CPI", "State 1 Industrial",
        "State 2 Cond. mean", "State 2 DXY", "State 2 GSCPI", "State 2 CPI", "State 2 Industrial",
        "p11", "p22"]


def print_section(title, separator):
    """Print the existing three-line heading in the original order."""
    print("\n" + separator)
    print(title)
    print(separator)


# Final-report and workbook formatting helpers.
def format_workbook(writer):
    """Apply consistent, readable formatting without changing worksheet values."""
    for sheet in writer.sheets.values():
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions
        sheet.sheet_view.showGridLines = False
        for cell in sheet[1]:
            cell.font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
            cell.fill = PatternFill("solid", fgColor="294D65")
            cell.alignment = Alignment(vertical="center", wrap_text=True)
        sheet.row_dimensions[1].height = 32
        for row in sheet.iter_rows(min_row=2):
            for cell in row:
                cell.font = Font(name="Arial", size=10, color="26323B")
                cell.alignment = Alignment(vertical="center", wrap_text=True)
                if cell.row % 2 == 0:
                    cell.fill = PatternFill("solid", fgColor="F2F5F7")
                if isinstance(cell.value, (float, np.floating)):
                    header = str(sheet.cell(1, cell.column).value).lower()
                    cell.number_format = "0.0000" if not any(
                        word in header for word in ("p_value", "q_value")) else "0.0000E+00"
                if hasattr(cell.value, "year") and hasattr(cell.value, "month"):
                    cell.number_format = "yyyy-mm-dd"
        for column in sheet.columns:
            values = [str(cell.value or "") for cell in column[:300]]
            width = max(len(line) for value in values for line in value.split("\n"))
            sheet.column_dimensions[column[0].column_letter].width = min(34, max(13, width + 2))
        if sheet.title == "Guide":
            sheet.column_dimensions["A"].width = 25
            sheet.column_dimensions["B"].width = 104
        for row in sheet.iter_rows(min_row=2):
            lines = 1
            for cell in row:
                width = sheet.column_dimensions[cell.column_letter].width
                text = "" if cell.value is None else str(cell.value)
                lines = max(lines, sum(max(1, math.ceil(len(line) / max(8, int(width))))
                                       for line in text.split("\n")))
            sheet.row_dimensions[row[0].row].height = max(18, 14 * lines)


def _number(value, places=4):
    if pd.isna(value):
        return "--"
    if np.isinf(value):
        return "infinite"
    return f"{value:,.{places}f}"


def _p_value(value):
    return "--" if pd.isna(value) else ("<0.001" if value < 0.001 else f"{value:.3f}")


def _table(frame):
    return '<div class="table-scroll">' + frame.to_html(index=False, border=0, escape=True) + '</div>'


def write_markov_report(output_dir, paper_tables, model_overview, coefficients,
                        state_mapping_tables, individual_tests, joint_tests,
                        model_warnings, probability_files, input_sha256,
                        appendix_diagnostics=None):
    """Export final results and return one HTML fragment for notebook display."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    intercepts = individual_tests.loc[individual_tests["test_type"] == "intercept"].copy()
    slopes = individual_tests.loc[individual_tests["test_type"] == "slope"].copy()
    selected = slopes.loc[slopes["q_value_bh"] < 0.05].copy()
    def bh_count(frame):
        values = frame.get("q_value_bh", pd.Series(dtype=float))
        return int((values < .05).sum()) if values.notna().any() else pd.NA
    summary = pd.DataFrame([
        {"Test family": "Conditional levels", "Tests": len(intercepts),
         "Raw p < 0.05": int((intercepts.p_value < .05).sum()), "BH q < 0.05": bh_count(intercepts)},
        {"Test family": "Individual slopes", "Tests": len(slopes),
         "Raw p < 0.05": int((slopes.p_value < .05).sum()),
         "BH q < 0.05": int((slopes.q_value_bh < .05).sum())},
        {"Test family": "Joint slopes", "Tests": len(joint_tests),
         "Raw p < 0.05": int((joint_tests.p_value < .05).sum()), "BH q < 0.05": bh_count(joint_tests)},
    ])
    notes = pd.DataFrame({"Item": ["Model", "Economic states", "Coefficient tests",
                                   "BH adjustment", "State duration", "Source consistency"],
                          "Definition": [
        "Two-state Markov regression; switching intercept, four slopes and variance; Nelder-Mead, maxiter=10000.",
        "RI: State 1 has higher absorption. RD: State 1 has faster relative recovery.",
        "Parentheses in coefficient tables contain within-state z-statistics; stars use raw p-values.",
        f"Three independent families: {len(intercepts)} conditional levels, {len(slopes)} individual slopes, {len(joint_tests)} joint slopes. Raw p-values are retained.",
        "Expected months = 1/(1-p_ss), using unrounded staying probabilities; this is distinct from RD.",
        "Tables, probabilities and Wald tests all use the same fitted model objects.",
    ]})
    workbook_path = output_dir / "Markov_results.xlsx"
    with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
        notes.to_excel(writer, sheet_name="Guide", index=False)
        model_overview.to_excel(writer, sheet_name="Model_overview", index=False)
        for shock, panels in paper_tables.items():
            for measure, table in panels.items():
                rows = []
                stats = model_overview.loc[(model_overview.shock == shock) & (model_overview.measure == measure)]
                for _, row in table.iterrows():
                    overview = stats.loc[stats.market == row["Market"]].iloc[0]
                    for state in (1, 2):
                        rows.append({"Market": row["Market"], "State": state,
                                     "Conditional mean": row[f"State {state} Cond. mean"],
                                     **{variable: row[f"State {state} {variable}"]
                                        for variable in ("DXY", "GSCPI", "CPI", "Industrial")},
                                     "Persistence": overview[f"p{state}{state}"],
                                     "Expected months": overview[f"expected_months_state{state}"]})
                pd.DataFrame(rows).to_excel(writer, sheet_name=f"{shock}_{measure}", index=False)
        coefficients.to_excel(writer, sheet_name="Coefficients_numeric", index=False)
        summary.to_excel(writer, sheet_name="Test_overview", index=False)
        slopes.to_excel(writer, sheet_name="Slope_differences", index=False)
        intercepts.to_excel(writer, sheet_name="Level_differences", index=False)
        joint_tests.to_excel(writer, sheet_name="Joint_slope_tests", index=False)
        pd.concat(state_mapping_tables.values(), ignore_index=True).to_excel(
            writer, sheet_name="State_mapping", index=False)
        if model_warnings:
            pd.DataFrame(model_warnings).to_excel(writer, sheet_name="Fit_warnings", index=False)
        if appendix_diagnostics is not None:
            for name, frame in appendix_diagnostics.items():
                frame.to_excel(writer, sheet_name=name[:31], index=False)
        format_workbook(writer)

    css = """
    .markov-report {font:14px/1.5 Arial,sans-serif;color:#26323b;max-width:1180px;margin:24px auto;padding:0 20px;}
    .markov-report h1 {font-size:26px;margin:0 0 6px;font-weight:600;}
    .markov-report h2 {font-size:19px;border-bottom:1px solid #cdd7df;padding-bottom:7px;margin-top:30px;}
    .markov-report h3 {font-size:15px;margin:20px 0 8px;}
    .markov-report .note {color:#52616d;font-size:12px;}
    .markov-report .status {background:#f0f4f7;padding:12px 16px;border-left:3px solid #527289;}
    .markov-report .attention {background:#fff4de;padding:12px 16px;border-left:3px solid #b38438;}
    .markov-report .table-scroll {overflow-x:auto;}
    .markov-report table {border-collapse:collapse;width:100%;font-size:12px;margin:8px 0 12px;white-space:nowrap;}
    .markov-report th {background:#e9eff3;font-weight:600;text-align:right;padding:8px 9px;border-bottom:1px solid #b8c8d3;}
    .markov-report td {text-align:right;padding:7px 9px;border-bottom:1px solid #e4eaee;}
    .markov-report th:first-child,.markov-report td:first-child {text-align:left;}
    .markov-report tbody tr:nth-child(even) {background:#f7f9fa;}
    .markov-report details {margin:12px 0;padding:10px 12px;border:1px solid #dce4e9;}
    .markov-report summary {cursor:pointer;font-weight:600;color:#294d65;}
    .markov-report a {color:#24557b;}
    @media print {.markov-report {max-width:none;padding:0;font-size:10pt;} .markov-report table {font-size:8pt;} details {break-inside:avoid;}}
    """
    converged = int(model_overview["converged"].fillna(False).sum())
    nobs = sorted(model_overview["nobs"].unique())
    parts = [f'<style>{css}</style><article class="markov-report">',
             '<h1>Markov regression: final results</h1>',
             '<p class="note">State 1: higher absorption for RI; faster relative recovery for RD. '
             'All tables and probabilities use the same estimation pass.</p>',
             f'<p class="status">{len(model_overview)} models · observations per model: '
             f'{", ".join(map(str, nobs))} · converged: {converged}/{len(model_overview)} · '
             f'individual slopes with BH q &lt; 0.05: {len(selected)}/{len(slopes)}</p>',
             f'<p><a href="{workbook_path.name}">Download the complete results workbook</a></p>']
    if converged != len(model_overview) or model_warnings:
        parts.append(f'<p class="attention">Review optimizer diagnostics: {len(model_warnings)} recorded '
                     'warnings. Convergence status is listed for every model in the workbook.</p>')
    for shock, panels in paper_tables.items():
        parts.append(f'<h2>{escape(shock)}: state levels and persistence</h2>')
        for measure, table in panels.items():
            overview = model_overview.loc[(model_overview.shock == shock) & (model_overview.measure == measure)]
            view = overview[["market", "mean_state1", "mean_state2", "p11", "p22",
                             "expected_months_state1", "expected_months_state2"]].copy()
            for column in ("mean_state1", "mean_state2", "p11", "p22"):
                view[column] = view[column].map(_number)
            for column in ("expected_months_state1", "expected_months_state2"):
                view[column] = view[column].map(lambda x: _number(x, 1))
            view.columns = ["Market", "State 1 mean", "State 2 mean", "p11", "p22", "E(D1), months", "E(D2), months"]
            parts.extend([f'<h3>{measure}</h3>', _table(view)])
            rows = []
            for _, row in table.iterrows():
                for state in (1, 2):
                    rows.append({"Market": row["Market"], "State": state,
                                 "Conditional mean": row[f"State {state} Cond. mean"],
                                 **{var: row[f"State {state} {var}"] for var in ("DXY", "GSCPI", "CPI", "Industrial")}})
            parts.extend([f'<details><summary>{measure}: full coefficient table</summary>',
                          '<p class="note">Estimates with raw within-state significance stars; z-statistics in parentheses.</p>',
                          _table(pd.DataFrame(rows)), '</details>'])
    parts.extend(['<h2>Cross-state inference</h2>', _table(summary.fillna("--")),
                  f'<p class="note">BH adjustment uses three separate families: {len(intercepts)} levels, '
                  f'{len(slopes)} individual slopes and {len(joint_tests)} joint slopes. '
                  'Raw p-values are also retained in the workbook.</p>',
                  '<h3>Individual slope differences retained after BH correction (q &lt; 0.05)</h3>'])
    view = selected[["shock", "measure", "market", "variable", "estimate_state1", "estimate_state2", "p_value", "q_value_bh"]].copy()
    for column in ("estimate_state1", "estimate_state2"):
        view[column] = view[column].map(_number)
    for column in ("p_value", "q_value_bh"):
        view[column] = view[column].map(_p_value)
    view.columns = ["Shock", "Measure", "Market", "Variable", "State 1", "State 2", "Raw p", "BH q"]
    parts.append(_table(view) if len(view) else '<p>No individual slope difference meets q &lt; 0.05.</p>')
    diagnostic_metadata = None
    if appendix_diagnostics is not None:
        diagnostics = appendix_diagnostics
        period = diagnostics["period_indicator"]
        alignment = diagnostics["weighted_alignment"]
        max_difference = float(alignment.absolute_difference.max()) if not alignment.empty else None
        parts.extend(['<h2>Appendix: date sensitivity</h2>',
            '<p class="note">Weighted regressions retain full-sample smoothed state probabilities and '
            'standardization. The primary level indicator starts in January 2021; alternatives start '
            'in March 2020 and February 2022. HAC uses six lags. States are not re-estimated.</p>'])
        if not period.empty:
            view = period[["shock", "measure", "market", "variable"]].copy()
            for state in (1, 2):
                view[f"State {state} with indicator"] = [f"{_number(b)} ({t:.2f})" for b, t in
                    zip(period[f"withD_s{state}"], period[f"withD_s{state}_t"])]
            view["Indicator, State 2"] = [f"{_number(b)} ({t:.2f})" for b, t in
                zip(period.indicator_s2, period.indicator_s2_t)]
            for tag in ("202003", "202101", "202202"):
                view[f"Ratio {tag[:4]}-{tag[4:]}"] = period[f"ratio_{tag}"].map(lambda x: _number(x, 2))
            parts.extend([f'<details><summary>Period-indicator results: {len(period)} selected slope differences</summary>',
                          _table(view), '<p class="note">Parentheses contain unadjusted HAC t-statistics. '
                          'Ratios are signed slope differences relative to the original Markov difference; '
                          'they are not explained-variance shares. No new cross-state tests or BH correction '
                          'are applied to these conditional sensitivity estimates.</p></details>'])
        parts.extend(['<details><summary>Supporting macrofinancial and subsample checks</summary>',
                      _table(diagnostics["macro_summary"]),
                      '<p class="note">The workbook contains the linear HAC benchmark, weighted pre-2021 '
                      'and excluding-2020 regressions, all three indicator dates, and full-sample alignment checks.</p>',
                      '</details>'])
        if max_difference is not None:
            parts.append(f'<p class="note">Maximum absolute slope difference between the no-indicator '
                         f'weighted regressions and the original Markov estimates: {max_difference:.6f}.</p>')
        diagnostic_metadata = dict(selected_slope_differences=len(period), primary_date="2021-01-01",
                                   indicator_dates=["2020-03-01", "2021-01-01", "2022-02-01"],
                                   hac_lags=6, probability_floor=1e-6,
                                   max_weighted_markov_slope_difference=max_difference,
                                   state_process_reestimated=False)
    parts.append('<h2>Probability files for R</h2><ul>')
    for path in probability_files.values():
        name = Path(path).name
        parts.append(f'<li><a href="{escape(name)}">{escape(name)}</a></li>')
    parts.append('</ul><p class="note">Exact estimates, convergence flags, all Wald tests and state mappings are retained in the workbook.</p></article>')
    fragment = "\n".join(parts)
    (output_dir / "Markov_results.html").write_text(
        '<!doctype html><html lang="en"><head><meta charset="utf-8">'
        '<meta name="viewport" content="width=device-width, initial-scale=1">'
        '<title>Markov regression: final results</title></head><body>' + fragment + '</body></html>', encoding="utf-8")
    (output_dir / "Markov_run_manifest.json").write_text(json.dumps({
        "input_sha256": input_sha256, "models": len(model_overview),
        "bh_family": "individual slopes (separate level and joint families)", "bh_family_size": len(slopes),
        "bh_significant_slopes_5pct": len(selected), "converged_models": converged,
        "recorded_fit_warnings": len(model_warnings), "one_estimation_pass": True,
        "bh_families": summary.fillna("not available").to_dict(orient="records"),
        "appendix_diagnostics": diagnostic_metadata,
    }, indent=2) + "\n", encoding="utf-8")
    # Notebook links resolve beside Markov.ipynb; the standalone HTML is inside output/.
    return fragment.replace('href="', 'href="output/')


# Optional manuscript-comparison helpers; no TeX file is required for analysis.
KEYS = ["shock", "measure", "market", "variable"]


NUMBER = re.compile(r"[+-]?(?:\d+\.\d+|\d+)")


def _region(text, label):
    start = text.index("\\label{" + label + "}")
    end = text.index("\\end{table}", start)
    return text[start:end]


def _numbers(cell):
    # The [-1pt] spacing command must not be interpreted as a table value.
    return NUMBER.findall(re.sub(r"\[[^\]]*\]", "", cell))


def _data_rows(text):
    rows = []
    for line in text.splitlines():
        if re.match(r"^(GPR|CPU|OPU)\s*&\s*(RI|RD)\s*&", line):
            rows.append([cell.strip() for cell in line.split(" & ")])
    if not rows:
        raise ValueError("No numerical rows were found in a requested TeX diagnostic table.")
    return rows


def verify_results(appendix_tex, workbook, main_tex=None):
    """Return one comparison record per displayed value or significance label."""
    text = Path(appendix_tex).read_text(encoding="utf-8")
    sheets = pd.read_excel(workbook, sheet_name=None)
    records = []

    def compare(table, case, field, displayed, computed, operator="rounded"):
        expected = float(displayed)
        computed = float(computed)
        decimals = len(displayed.split(".")[1]) if "." in displayed else 0
        tolerance = 0.5 * 10 ** (-decimals) + 1e-12
        if operator == "lt":
            match = np.isfinite(computed) and computed < expected
        elif operator == "le":
            match = np.isfinite(computed) and computed <= expected + 1e-12
        else:
            match = np.isfinite(computed) and abs(computed - expected) <= tolerance
        records.append(dict(table=table, case=case, field=field, expected_display=displayed,
                            computed=computed, difference=computed - expected,
                            tolerance=tolerance if operator == "rounded" else np.nan,
                            comparison=operator, match=bool(match)))

    specifications = [
        ("tab:period_indicator", "period_indicator", {
            4: ["ms_s1"], 5: ["ms_s2"], 6: ["withD_s1", "withD_s1_t"],
            7: ["withD_s2", "withD_s2_t"], 8: ["indicator_s2", "indicator_s2_t"],
            9: ["ratio_202003"], 10: ["ratio_202101"], 11: ["ratio_202202"]}),
        ("tab:macro_subsample_diagnostics", "subsample_checks", {
            4: ["linear", "linear_p"], 5: ["ms_s1"], 6: ["ms_s2"],
            7: ["pre2021_s1", "pre2021_s1_t"], 8: ["pre2021_s2", "pre2021_s2_t"],
            9: ["excl2020_s1", "excl2020_s1_t"], 10: ["excl2020_s2", "excl2020_s2_t"]}),
    ]
    for label, sheet, fields in specifications:
        calculated = sheets[sheet].set_index(KEYS)
        seen = set()
        for cells in _data_rows(_region(text, label)):
            key = tuple(cells[:4])
            seen.add(key)
            if key not in calculated.index:
                raise ValueError(f"A manuscript case is absent from the generated results: {key}")
            row = calculated.loc[key]
            for column, names in fields.items():
                values = _numbers(cells[column])
                if len(values) != len(names):
                    raise ValueError(f"Unexpected numerical format: {label}/{key}/{cells[column]}")
                for field, value in zip(names, values):
                    operator = "lt" if field == "linear_p" and "<" in cells[column] else "rounded"
                    compare(label, "-".join(key), field, value, row[field], operator)
        if seen != set(calculated.index):
            raise ValueError(f"The generated and manuscript case sets differ for {label}.")

    macro = _region(text, "tab:macro_diagnostic_summary")
    counts = sheets["macro_summary"].set_index("diagnostic")["value"]
    for line in macro.splitlines():
        if "greater fitted" in line or "same comparison, omitting" in line:
            field = "greater_dispersion_full" if "greater fitted" in line else "greater_dispersion_excl2020"
            value, total = re.search(r"&\s*(\d+)\s+of\s+(\d+)", line).groups()
            compare("tab:macro_diagnostic_summary", "all models", field, value, counts[field])
            compare("tab:macro_diagnostic_summary", "all models", "regressions", total, len(sheets["fitted_dispersion"]))
        elif "Within-state slope detections" in line:
            for state in (1, 2):
                value = re.search(r"State~" + str(state) + r":\s*(\d+)", line).group(1)
                field = f"within_state{state}_detections"
                compare("tab:macro_diagnostic_summary", "all models", field, value, counts[field])
        elif "Linear OLS--HAC coefficient detections" in line:
            for variable in ("DXY", "GSCPI", "CPI", "Industrial"):
                value = re.search(r"\b" + re.escape(variable) + r":\s*(\d+)", line).group(1)
                field = f"linear_{variable}_detections"
                compare("tab:macro_diagnostic_summary", "all models", field, value, counts[field])

    compare("tab:period_indicator", "all selected slopes", "maximum_weighted_alignment_error",
            "0.002", sheets["weighted_alignment"].absolute_difference.max(), "le")
    for field, value in {"n_full": 166, "n_pre2021": 131, "n_excl2020": 156,
                         "n_indicator_2021-01": 35}.items():
        for _, row in sheets["diagnostic_samples"].iterrows():
            compare("sample sizes", "-".join(str(row[k]) for k in KEYS[:3]), field, str(value), row[field])

    if main_tex is not None:
        main = Path(main_tex).read_text(encoding="utf-8")
        restrictions = [("intercept", r"\mu_1=\mu_2", sheets["Level_differences"]),
                        ("joint slopes", r"\bm{\beta}_1=\bm{\beta}_2", sheets["Joint_slope_tests"])]
        for shock in ("GPR", "CPU", "OPU"):
            body = _region(main, f"tab:ms_{shock.lower()}")
            panel = body[body.index("Panel C: Wald tests"):]
            for measure in ("RI", "RD"):
                for restriction, expression, frame in restrictions:
                    label = measure + ": $" + expression + "$"
                    tail = panel.split(label, 1)[1]
                    cells = tail[:tail.index(r"\\")].split("&")[1:]
                    if len(cells) != 5:
                        raise ValueError(f"Expected five markets in Panel C: {shock}/{measure}/{restriction}")
                    for market, cell in zip(("OIL", "NG", "COAL", "ERIX", "EUA"), cells):
                        row = frame.loc[(frame.shock == shock) & (frame.measure == measure) & (frame.market == market)].iloc[0]
                        case = f"{shock}-{measure}-{market}-{restriction}"
                        compare("main Panel C", case, "wald_chi2", _numbers(cell)[0], row.wald_chi2)
                        expected = re.search(r"\^\{(\*+)\}", cell)
                        expected = expected.group(1) if expected else ""
                        computed = "" if pd.isna(row.stars_q) else row.stars_q
                        records.append(dict(table="main Panel C", case=case, field="BH significance stars",
                                            expected_display=expected, computed=computed,
                                            difference=np.nan, tolerance=np.nan, comparison="exact",
                                            match=expected == computed))
    return pd.DataFrame(records)


def verify_manuscript_files(appendix_tex, workbook, main_tex=None):
    """Write the optional TeX comparison files and fail if any entry differs."""
    appendix_tex = Path(appendix_tex).expanduser().resolve()
    workbook = Path(workbook).expanduser().resolve()
    main_tex = Path(main_tex).expanduser().resolve() if main_tex is not None else None
    comparisons = verify_results(appendix_tex, workbook, main_tex)
    destination = workbook.parent
    comparisons.to_csv(destination / "Appendix_verification.csv", index=False)
    sources = [appendix_tex, *([main_tex] if main_tex is not None else [])]
    summary = dict(comparisons=len(comparisons), matched=int(comparisons.match.sum()),
                   mismatched=int((~comparisons.match).sum()),
                   by_table=comparisons.groupby("table").agg(
                       comparisons=("match", "size"), matched=("match", "sum")).to_dict("index"),
                   source_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sources})
    (destination / "Appendix_verification.json").write_text(
        json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    if not comparisons.match.all():
        raise AssertionError(f"{summary['mismatched']} manuscript entries differ; "
                             "see output/Appendix_verification.csv.")
    return comparisons, summary


## 2. Estimate final models and export R-ready probabilities

Set `SELECTED_UNCERTAINTY`, `STATE_TO_PLOT` and reporting options below.


In [2]:
# 0) Basic settings

# Choose one uncertainty index:
#     "GPR" -> Var6  to Var10
#     "CPU" -> Var14 to Var18
#     "OPU" -> Var22 to Var26
#     "ALL" -> estimate all 30 models and export one probability workbook per index
SELECTED_UNCERTAINTY = "ALL"

# Probability selected for the R plotting column `prob_selected`:
#     1 -> economically defined State 1
#     2 -> economically defined State 2
STATE_TO_PLOT = 1

# True  -> export smoothed marginal probabilities
# False -> export filtered marginal probabilities
USE_SMOOTHED_PROBABILITIES = True

# False: display only the final report after all model and test cells finish.
# True: also print each complete statsmodels summary during estimation.
PRINT_STATSMODELS_SUMMARY = False
VERBOSE = False  # Optional progress headings; final results are always in the last cell.

# Whether to export the probability workbook(s).
EXPORT_PROBABILITY_XLSX = True

markets = ["OIL", "NG", "COAL", "ERIX", "EUA"]

uncertainty_variable_maps = {
    "GPR": {'OIL': 'Var6', 'NG': 'Var7', 'COAL': 'Var8', 'ERIX': 'Var9', 'EUA': 'Var10'},
    "CPU": {'OIL': 'Var14', 'NG': 'Var15', 'COAL': 'Var16', 'ERIX': 'Var17', 'EUA': 'Var18'},
    "OPU": {'OIL': 'Var22', 'NG': 'Var23', 'COAL': 'Var24', 'ERIX': 'Var25', 'EUA': 'Var26'}
}

exog_cols = ["DXY", "GSCPI", "CPI", "Industrial"]
DATE_COLUMN = "date"

# Input files
y_path = RESILIENCE_FILE

x_path = MACRO_DATA_FILE

# Probability workbooks are saved here.
OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR

# Probability-specific helpers; shared model utilities are in the setup cell.


def validate_state_to_plot(state_to_plot):
    """Validate the reported-state choice used for `prob_selected`."""
    if state_to_plot not in (1, 2):
        raise ValueError("STATE_TO_PLOT must be 1 or 2.")


def extract_probability_array(probability_object, regime_index, nobs):
    """
    Extract one raw-regime marginal-probability series as a NumPy array.

    This handles both pandas DataFrames and NumPy arrays returned by
    different statsmodels versions.
    """
    if isinstance(probability_object, pd.DataFrame):
        if regime_index in probability_object.columns:
            values = probability_object[regime_index].to_numpy(dtype=float)
        elif str(regime_index) in probability_object.columns:
            values = probability_object[str(regime_index)].to_numpy(dtype=float)
        else:
            values = probability_object.iloc[:, regime_index].to_numpy(dtype=float)

    elif isinstance(probability_object, pd.Series):
        if regime_index != 0:
            raise ValueError("A one-dimensional probability Series cannot supply "
                f"regime {regime_index}.")
        values = probability_object.to_numpy(dtype=float)

    else:
        array = np.asarray(probability_object, dtype=float)

        if array.ndim != 2:
            raise ValueError(f"Unexpected probability array shape: {array.shape}.")

        if array.shape[0] == nobs:
            values = array[:, regime_index]
        elif array.shape[1] == nobs:
            values = array[regime_index, :]
        else:
            raise ValueError(f"Unexpected probability array shape: {array.shape}; "
                f"one dimension must equal nobs={nobs}.")

    if len(values) != nobs:
        raise ValueError(f"Probability series has length {len(values)}, "
            f"but the model sample contains {nobs} observations.")

    return np.asarray(values, dtype=float)


def build_probability_table(result, model_data, uncertainty_name, measure_name, market,
    state_1_raw, state_2_raw, state_to_plot, use_smoothed,):
    """Build the long-form probability output for one fitted model."""
    probability_object = (result.smoothed_marginal_probabilities
        if use_smoothed
        else result.filtered_marginal_probabilities)

    nobs = len(model_data)
    prob_raw_0 = extract_probability_array(probability_object=probability_object, regime_index=0,
        nobs=nobs)
    prob_raw_1 = extract_probability_array(probability_object=probability_object, regime_index=1,
        nobs=nobs)

    raw_probability_map = {0: prob_raw_0, 1: prob_raw_1}

    prob_state_1 = raw_probability_map[state_1_raw]
    prob_state_2 = raw_probability_map[state_2_raw]

    if state_to_plot == 1:
        selected_raw_regime = state_1_raw
        prob_selected = prob_state_1
    else:
        selected_raw_regime = state_2_raw
        prob_selected = prob_state_2

    probability_type = "smoothed" if use_smoothed else "filtered"

    return pd.DataFrame({"date": model_data[DATE_COLUMN].to_numpy(), "shock": uncertainty_name,
        "measure": measure_name, "market": market,
        "metric_value": model_data["y"].to_numpy(dtype=float),
        "y_centered": model_data["y_centered"].to_numpy(dtype=float),
        "probability_type": probability_type, "prob_raw_regime0": prob_raw_0,
        "prob_raw_regime1": prob_raw_1, "prob_state1": prob_state_1, "prob_state2": prob_state_2,
        "state_to_plot": state_to_plot, "selected_raw_regime": selected_raw_regime,
        "prob_selected": prob_selected})


def build_state_mapping_row(result, uncertainty_name, measure_name, market, y_mean, regime_results,
    state_1_raw, state_2_raw, state_to_plot, use_smoothed,):
    """Build one state-mapping record for checking R plot labels."""
    selected_raw_regime = (state_1_raw if state_to_plot == 1 else state_2_raw)

    if measure_name == "RI":
        state_1_definition = "higher original-scale conditional mean"
        state_2_definition = "lower original-scale conditional mean"
    else:
        state_1_definition = "lower original-scale conditional mean"
        state_2_definition = "higher original-scale conditional mean"

    mle_retvals = getattr(result, "mle_retvals", {})
    converged = (mle_retvals.get("converged", np.nan)
        if isinstance(mle_retvals, dict)
        else np.nan)

    return {"shock": uncertainty_name, "measure": measure_name, "market": market,
        "probability_type": "smoothed" if use_smoothed else "filtered",
        "state1_definition": state_1_definition, "state2_definition": state_2_definition,
        "state_to_plot": state_to_plot, "selected_raw_regime": selected_raw_regime,
        "y_mean": float(y_mean), "const0_centered": float(regime_results[0]["centered_const"]),
        "const1_centered": float(regime_results[1]["centered_const"]), "const0_original": float(
            regime_results[0]["conditional_mean"]), "const1_original": float(
            regime_results[1]["conditional_mean"]), "state1_raw": int(state_1_raw),
        "state2_raw": int(state_2_raw), "state1_original_intercept": float(
            regime_results[state_1_raw]["conditional_mean"]), "state2_original_intercept": float(
            regime_results[state_2_raw]["conditional_mean"]), "nobs": int(result.nobs),
        "converged": converged}


def export_probability_workbook(uncertainty_name, probability_table, state_mapping_table,
    output_dir, state_to_plot,):
    """Export R-ready probabilities and state mapping to one XLSX file."""
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / (f"{uncertainty_name.lower()}_ms_state_probabilities_"
        f"state{state_to_plot}.xlsx")

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        probability_table.to_excel(writer, sheet_name="probabilities_long", index=False)
        state_mapping_table.to_excel(writer, sheet_name="state_mapping", index=False)

        # Shared styling changes presentation only; R worksheet schemas stay fixed.
        format_workbook(writer)
        workbook = writer.book

        probability_sheet = writer.sheets["probabilities_long"]
        probability_sheet.freeze_panes = "A2"
        probability_sheet.auto_filter.ref = probability_sheet.dimensions
        probability_sheet.column_dimensions["A"].width = 13

        for cell in probability_sheet["A"][1:]:
            cell.number_format = "yyyy-mm-dd"

        mapping_sheet = writer.sheets["state_mapping"]
        mapping_sheet.freeze_panes = "A2"
        mapping_sheet.auto_filter.ref = mapping_sheet.dimensions

        # Keep all worksheets visible and leave the workbook active on
        # probabilities_long, which is the sheet used directly in R.
        workbook.active = workbook.sheetnames.index("probabilities_long")

    return output_path

# 2) Read and validate data

if not y_path.exists():
    raise FileNotFoundError(f"Dependent-variable file not found: {y_path}")

if not x_path.exists():
    raise FileNotFoundError(f"Explanatory-variable file not found: {x_path}")

r_data = pd.read_excel(y_path)
x_data = pd.read_csv(x_path)

required_y_columns = {"Variable", "RP", "RD"}
missing_y_columns = sorted(required_y_columns - set(r_data.columns))

if missing_y_columns:
    raise ValueError(f"resilience_metrics.xlsx is missing columns: {missing_y_columns}")

missing_exog_cols = [column
    for column in exog_cols
    if column not in x_data.columns]

if missing_exog_cols:
    raise ValueError(f"The explanatory-variable file is missing columns: "
        f"{missing_exog_cols}")

if DATE_COLUMN not in x_data.columns:
    raise ValueError(f"The explanatory-variable file must contain a '{DATE_COLUMN}' "
        "column so the exported probabilities can be plotted in R.")

# Do not drop missing observations here. Missing y, X, and date values are
# removed jointly within each market model so observations remain aligned.
Z_source = (x_data[exog_cols]
    .apply(pd.to_numeric, errors="coerce")
    .reset_index(drop=True))

date_source = (pd.to_datetime(x_data[DATE_COLUMN], errors="coerce", utc=True)
    .dt.tz_convert(None)
    .reset_index(drop=True))

selected_uncertainties = validate_uncertainty_selection(SELECTED_UNCERTAINTY)
validate_state_to_plot(STATE_TO_PLOT)

# 3) Result containers

# Complete statsmodels results:
# results[uncertainty][measure][market]
results = {}

# Paper-style tables:
# paper_tables[uncertainty][measure]
paper_tables = {}

# Long probability outputs:
# probability_tables[uncertainty]
probability_tables = {}

# State-label checks:
# state_mapping_tables[uncertainty]
state_mapping_tables = {}

# Exported workbook paths:
# exported_probability_files[uncertainty]
exported_probability_files = {}

display_columns = paper_display_columns()
model_overview_rows = []
coefficient_rows = []
model_warnings = []

# 4) Estimate selected uncertainty index / indices

for uncertainty_name in selected_uncertainties:

    variable_map = uncertainty_variable_maps[uncertainty_name]
    validate_variable_codes(r_data=r_data, uncertainty_name=uncertainty_name,
        variable_map=variable_map)

    # Build RI and RD data for this uncertainty index.
    df_RI = build_dependent_dataframe(r_data=r_data, value_column="RP", variable_map=variable_map)

    df_RD = build_dependent_dataframe(r_data=r_data, value_column="RD", variable_map=variable_map)

    # Align the overall positional lengths for this uncertainty index.
    n = min(len(df_RI), len(df_RD), len(Z_source), len(date_source))

    if n == 0:
        raise ValueError(f"No observations are available for {uncertainty_name}.")

    df_RI = df_RI.iloc[:n].reset_index(drop=True)
    df_RD = df_RD.iloc[:n].reset_index(drop=True)
    Z_full = Z_source.iloc[:n].reset_index(drop=True)
    dates_full = date_source.iloc[:n].reset_index(drop=True)

    datasets = {"RI": df_RI, "RD": df_RD}

    results[uncertainty_name] = {"RI": {}, "RD": {}}

    paper_tables[uncertainty_name] = {}

    uncertainty_probability_parts = []
    uncertainty_mapping_rows = []

    if VERBOSE:
        print_section(f'UNCERTAINTY INDEX: {uncertainty_name}', '#' * 120)

    for measure_name, dependent_data in datasets.items():

        if VERBOSE:
            print_section(f'Estimating {uncertainty_name} - {measure_name}', '#' * 100)

        paper_rows = []

        for market in markets:

            if VERBOSE:
                print_section(f'{uncertainty_name} - {measure_name} - {market}', '=' * 90)

            # 4.1 Jointly align date, y, and X

            model_data = pd.concat([dates_full.rename(DATE_COLUMN),
                    dependent_data[market].rename("y"), Z_full], axis=1)

            model_data = (model_data
                .dropna(subset=[DATE_COLUMN, "y"] + exog_cols)
                .sort_values(DATE_COLUMN)
                .reset_index(drop=True))

            if model_data.empty:
                raise ValueError(f"No valid observations for "
                    f"{uncertainty_name} - {measure_name} - {market}.")

            y_original = model_data["y"].astype(float)
            Z = model_data[exog_cols].astype(float)

            # 4.2 Center dependent variable

            y_mean = float(y_original.mean())
            y_centered = y_original - y_mean
            model_data["y_centered"] = y_centered.to_numpy(dtype=float)

            # 4.3 Estimate Markov-switching regression

            model = sm.tsa.MarkovRegression(endog=y_centered, exog=Z, k_regimes=2, trend="c",
                switching_trend=True, switching_exog=True, switching_variance=True)

            with warnings.catch_warnings(record=True) as captured_warnings:
                warnings.simplefilter("always")
                result = model.fit(method="nm", maxiter=10000, disp=False)
            model_warnings.extend({"shock": uncertainty_name, "measure": measure_name,
                                   "market": market, "category": warning.category.__name__,
                                   "message": str(warning.message)} for warning in captured_warnings)

            results[uncertainty_name][measure_name][market] = result

            if PRINT_STATSMODELS_SUMMARY:
                print(result.summary())

            # 4.4 Obtain named estimates and standard errors

            parameter_names = get_parameter_names(result)

            params = make_named_series(result.params, parameter_names)

            standard_errors = make_named_series(result.bse, parameter_names)

            # 4.5 Restore regime means to the original scale

            regime_results = {}

            for raw_regime in [0, 1]:

                const_name = find_regime_parameter(names=parameter_names, variable="const",
                    regime=raw_regime, exog_columns=exog_cols)

                centered_const = float(params[const_name])
                const_se = float(standard_errors[const_name])
                conditional_mean = y_mean + centered_const

                mean_z, mean_p = calculate_z_p(conditional_mean, const_se)

                regime_results[raw_regime] = {"centered_const": centered_const,
                    "conditional_mean": conditional_mean, "conditional_mean_se": const_se,
                    "conditional_mean_z": mean_z, "conditional_mean_p": mean_p, "coefficients": {}}

                for variable in exog_cols:

                    coefficient_name = find_regime_parameter(names=parameter_names,
                        variable=variable, regime=raw_regime, exog_columns=exog_cols)

                    estimate = float(params[coefficient_name])
                    standard_error = float(standard_errors[coefficient_name])

                    z_value, p_value = calculate_z_p(estimate, standard_error)

                    regime_results[raw_regime]["coefficients"][variable] = {"estimate": estimate,
                        "standard_error": standard_error, "z": z_value, "p": p_value}

            # 4.6 Extract raw-regime persistence probabilities

            p00_name = find_parameter(parameter_names, "p[0->0]")

            p10_name = find_parameter(parameter_names, "p[1->0]")

            if p00_name is None or p10_name is None:
                raise ValueError(f"Cannot find transition parameters for "
                    f"{uncertainty_name} - {measure_name} - {market}.\n"
                    f"Available parameters:\n{parameter_names}")

            raw_persistence = {0: {"estimate": float(params[p00_name]),
                    "standard_error": float(standard_errors[p00_name])}, 1: {
                    "estimate": float(1.0 - params[p10_name]),
                    "standard_error": float(standard_errors[p10_name])}}

            # 4.7 Relabel regimes by economic interpretation

            state_1_raw, state_2_raw = relabel_regimes(measure_name=measure_name,
                regime_results=regime_results)

            # 4.8 Build one paper-style output row

            row = {"Market": market}

            state_1_result = regime_results[state_1_raw]
            state_2_result = regime_results[state_2_raw]

            row["State 1 Cond. mean"] = paper_format(state_1_result["conditional_mean"],
                state_1_result["conditional_mean_z"], state_1_result["conditional_mean_p"])

            for variable in exog_cols:
                coefficient = state_1_result["coefficients"][variable]

                row[f"State 1 {variable}"] = paper_format(coefficient["estimate"],
                    coefficient["z"], coefficient["p"])

            row["State 2 Cond. mean"] = paper_format(state_2_result["conditional_mean"],
                state_2_result["conditional_mean_z"], state_2_result["conditional_mean_p"])

            for variable in exog_cols:
                coefficient = state_2_result["coefficients"][variable]

                row[f"State 2 {variable}"] = paper_format(coefficient["estimate"],
                    coefficient["z"], coefficient["p"])

            # p11: probability of remaining in economically defined State 1
            p11 = raw_persistence[state_1_raw]["estimate"]
            p11_se = raw_persistence[state_1_raw]["standard_error"]
            p11_z, p11_p = calculate_z_p(p11, p11_se)

            row["p11"] = paper_format(p11, p11_z, p11_p)

            # p22: probability of remaining in economically defined State 2
            p22 = raw_persistence[state_2_raw]["estimate"]
            p22_se = raw_persistence[state_2_raw]["standard_error"]
            p22_z, p22_p = calculate_z_p(p22, p22_se)

            row["p22"] = paper_format(p22, p22_z, p22_p)

            # Retain raw-regime mapping for verification, but do not display it
            # in the final paper-style tables.
            row["State 1 raw regime"] = state_1_raw
            row["State 2 raw regime"] = state_2_raw

            paper_rows.append(row)

            # 4.9 Reuse this fitted result to export probabilities

            probability_part = build_probability_table(result=result, model_data=model_data,
                uncertainty_name=uncertainty_name, measure_name=measure_name, market=market,
                state_1_raw=state_1_raw, state_2_raw=state_2_raw, state_to_plot=STATE_TO_PLOT,
                use_smoothed=USE_SMOOTHED_PROBABILITIES)
            uncertainty_probability_parts.append(probability_part)

            mapping_row = build_state_mapping_row(result=result, uncertainty_name=uncertainty_name,
                measure_name=measure_name, market=market, y_mean=y_mean,
                regime_results=regime_results, state_1_raw=state_1_raw, state_2_raw=state_2_raw,
                state_to_plot=STATE_TO_PLOT, use_smoothed=USE_SMOOTHED_PROBABILITIES)
            uncertainty_mapping_rows.append(mapping_row)

            # Full-precision final summaries use this exact fitted object.
            model_overview_rows.append({
                "shock": uncertainty_name, "measure": measure_name, "market": market,
                "nobs": int(result.nobs), "converged": mapping_row["converged"],
                "mean_state1": state_1_result["conditional_mean"],
                "mean_state2": state_2_result["conditional_mean"], "p11": p11, "p22": p22,
                "expected_months_state1": 1.0 / (1.0 - p11) if p11 < 1 else np.inf,
                "expected_months_state2": 1.0 / (1.0 - p22) if p22 < 1 else np.inf,
                "log_likelihood": float(result.llf), "aic": float(result.aic), "bic": float(result.bic)})
            for state, estimates in ((1, state_1_result), (2, state_2_result)):
                coefficient_rows.append({"shock": uncertainty_name, "measure": measure_name,
                    "market": market, "state": state, "term": "conditional_mean",
                    "estimate": estimates["conditional_mean"],
                    "standard_error": estimates["conditional_mean_se"],
                    "z": estimates["conditional_mean_z"], "p_value": estimates["conditional_mean_p"]})
                for variable, coefficient in estimates["coefficients"].items():
                    coefficient_rows.append({"shock": uncertainty_name, "measure": measure_name,
                        "market": market, "state": state, "term": variable, **coefficient})

        paper_tables[uncertainty_name][measure_name] = pd.DataFrame(paper_rows)

    # 6) Combine and export probability outputs

    # Preserve the same row order as the original probability-export code:
    # OIL-RI, OIL-RD, NG-RI, NG-RD, ..., EUA-RI, EUA-RD.
    # Do not sort market names alphabetically.
    market_order = {market: position for position, market in enumerate(markets)}
    measure_order = {"RI": 0, "RD": 1}

    probability_tables[uncertainty_name] = pd.concat(uncertainty_probability_parts,
        ignore_index=True)

    probability_tables[uncertainty_name]["_market_order"] = (
        probability_tables[uncertainty_name]["market"].map(market_order))
    probability_tables[uncertainty_name]["_measure_order"] = (
        probability_tables[uncertainty_name]["measure"].map(measure_order))

    probability_tables[uncertainty_name] = (probability_tables[uncertainty_name]
        .sort_values(["_market_order", "_measure_order", "date"], kind="stable")
        .drop(columns=["_market_order", "_measure_order"])
        .reset_index(drop=True))

    state_mapping_tables[uncertainty_name] = pd.DataFrame(uncertainty_mapping_rows)

    state_mapping_tables[uncertainty_name]["_market_order"] = (
        state_mapping_tables[uncertainty_name]["market"].map(market_order))
    state_mapping_tables[uncertainty_name]["_measure_order"] = (
        state_mapping_tables[uncertainty_name]["measure"].map(measure_order))

    state_mapping_tables[uncertainty_name] = (state_mapping_tables[uncertainty_name]
        .sort_values(["_market_order", "_measure_order"], kind="stable")
        .drop(columns=["_market_order", "_measure_order"])
        .reset_index(drop=True))

    if EXPORT_PROBABILITY_XLSX:
        exported_probability_files[uncertainty_name] = (export_probability_workbook(
                uncertainty_name=uncertainty_name,
                probability_table=probability_tables[uncertainty_name],
                state_mapping_table=state_mapping_tables[uncertainty_name], output_dir=OUTPUT_DIR,
                state_to_plot=STATE_TO_PLOT))

# The report and both test cells consume the same final results dictionary.
model_overview = pd.DataFrame(model_overview_rows)
coefficient_table = pd.DataFrame(coefficient_rows)
# Standardize coefficient-field names; the numeric estimates stay unchanged.
coefficient_table["p_value"] = coefficient_table["p_value"].fillna(coefficient_table["p"])
coefficient_table = coefficient_table.drop(columns="p")


## 3. Individual slope equality (raw p-values)


In [3]:
# Individual cross-state slope equality: full-covariance Wald tests.
# Run after the probability-export estimation cell; preserve economic state labels.
import numpy as np
import pandas as pd
from scipy.stats import chi2

EXPORT_EQUALITY_XLSX = True
EQUALITY_OUTPUT_PATH = OUTPUT_DIR / "cross_state_equality_tests.xlsx"


def find_regime_coef_index(names, variable, regime):
    """Locate the parameter index of one regime coefficient.

    Supports both statsmodels naming formats:
        DXY[0], GSCPI[1], ...
    and:
        x1[0], x2[1], ...
    """
    if variable == "const":
        candidates = [f"const[{regime}]"]
    else:
        candidates = [f"{variable}[{regime}]"]
        position = exog_cols.index(variable) + 1
        candidates.append(f"x{position}[{regime}]")
    for i, nm in enumerate(names):
        if str(nm) in candidates:
            return i
    return None


def significance_stars_eq(p_value):
    if pd.isna(p_value):
        return ""
    if p_value < 0.01:
        return "***"
    if p_value < 0.05:
        return "**"
    if p_value < 0.10:
        return "*"
    return ""

equality_rows = []

for uncertainty_name, shock_results in results.items():
    for measure_name, market_results in shock_results.items():
        for market, result in market_results.items():

            names = list(result.params.index) if hasattr(result.params, "index") else list(result.model.param_names)
            params_vec = np.asarray(result.params, dtype=float)
            cov = np.asarray(result.cov_params(), dtype=float)

            # Identify the raw regimes corresponding to State 1 and State 2
            # under the economic labeling used in the paper.
            const_idx = {r: find_regime_coef_index(names, "const", r) for r in (0, 1)}
            if const_idx[0] is None or const_idx[1] is None:
                continue
            cond_means = {r: params_vec[const_idx[r]] for r in (0, 1)}
            if measure_name == "RI":
                state1_raw = max((0, 1), key=lambda r: cond_means[r])
            else:
                state1_raw = min((0, 1), key=lambda r: cond_means[r])
            state2_raw = 1 - state1_raw

            for variable in exog_cols:
                i1 = find_regime_coef_index(names, variable, state1_raw)
                i2 = find_regime_coef_index(names, variable, state2_raw)
                if i1 is None or i2 is None:
                    continue

                beta1 = params_vec[i1]
                beta2 = params_vec[i2]

                # Wald test of beta_{s1} - beta_{s2} = 0.
                R = np.zeros((1, len(params_vec)))
                R[0, i1] = 1.0
                R[0, i2] = -1.0
                diff = (R @ params_vec).item()
                var = (R @ cov @ R.T).item()

                if var <= 0 or not np.isfinite(var):
                    wald = np.nan
                    p_value = np.nan
                else:
                    wald = diff ** 2 / var
                    p_value = float(chi2.sf(wald, df=1))

                equality_rows.append({"shock": uncertainty_name, "measure": measure_name,
                    "market": market, "variable": variable, "beta_state1": round(beta1, 4),
                    "beta_state2": round(beta2, 4), "sign_opposite": (beta1 * beta2) < 0,
                    "diff": round(diff, 4), "se_diff": round(var ** 0.5, 4),
                    "wald": round(wald, 3) if pd.notna(wald) else np.nan, "p_value": p_value,
                    "stars": significance_stars_eq(p_value)})

equality_table = pd.DataFrame(equality_rows)

if EXPORT_EQUALITY_XLSX:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(EQUALITY_OUTPUT_PATH, engine="openpyxl") as writer:
        equality_table.to_excel(writer, sheet_name="equality_tests", index=False)
        sheet = writer.sheets["equality_tests"]
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions
        format_workbook(writer)


## 4. Conditional-level, slope and joint tests

Each restriction type has its own BH family.


In [4]:
# Comprehensive level, individual-slope, and joint-slope Wald tests.
# Three separate BH families follow main_J1108.tex and Supplementary_Appendix.tex.
import numpy as np
import pandas as pd
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

EXPORT_COMPREHENSIVE_TESTS_XLSX = True

# Keep the user's existing output convention when OUTPUT_DIR is available.
# Otherwise write to the current working directory.
try:
    _test_output_dir = Path(OUTPUT_DIR)
except NameError:
    _test_output_dir = Path.cwd()

COMPREHENSIVE_TEST_OUTPUT_PATH = (_test_output_dir / "cross_state_comprehensive_tests.xlsx")


def _find_regime_parameter_index(names, variable, regime):
    """Return the parameter index for a regime-specific intercept or slope."""
    if variable == "const":
        candidates = [f"const[{regime}]"]
    else:
        candidates = [f"{variable}[{regime}]"]
        position = exog_cols.index(variable) + 1
        candidates.append(f"x{position}[{regime}]")

    for i, name in enumerate(names):
        if str(name) in candidates:
            return i
    return None


def _wald_linear_restriction(params, cov, R):
    """Wald test for H0: R @ params = 0."""
    params = np.asarray(params, dtype=float)
    cov = np.asarray(cov, dtype=float)
    R = np.atleast_2d(np.asarray(R, dtype=float))

    restriction = R @ params
    middle = R @ cov @ R.T

    if not (np.all(np.isfinite(restriction))
        and np.all(np.isfinite(middle))):
        return np.nan, np.nan, np.full(len(restriction), np.nan), len(restriction)

    # pinv is preferable here to an unconditional inverse because estimated
    # covariance matrices can be numerically close to singular.
    middle_inv = np.linalg.pinv(middle)
    statistic = float(restriction.T @ middle_inv @ restriction)
    df = int(np.linalg.matrix_rank(R))

    if df <= 0 or statistic < 0 or not np.isfinite(statistic):
        return np.nan, np.nan, restriction, df

    p_value = float(chi2.sf(statistic, df=df))
    return statistic, p_value, restriction, df


def _stars(p_value):
    if pd.isna(p_value):
        return ""
    if p_value <= 0.01:
        return "***"
    if p_value <= 0.05:
        return "**"
    if p_value <= 0.10:
        return "*"
    return ""

individual_rows = []
joint_rows = []

for uncertainty_name, shock_results in results.items():
    for measure_name, market_results in shock_results.items():
        for market, result in market_results.items():

            names = (list(result.params.index)
                if hasattr(result.params, "index")
                else list(result.model.param_names))
            params = np.asarray(result.params, dtype=float)
            cov = np.asarray(result.cov_params(), dtype=float)

            # A) Recover the economic State 1 / State 2 labels
            const_idx = {r: _find_regime_parameter_index(names, "const", r)
                for r in (0, 1)}
            if const_idx[0] is None or const_idx[1] is None:
                raise ValueError(f"Cannot find both regime intercepts for "
                    f"{uncertainty_name}-{measure_name}-{market}.")

            raw_const = {r: params[const_idx[r]] for r in (0, 1)}

            if measure_name == "RI":
                state1_raw = max((0, 1), key=lambda r: raw_const[r])
            elif measure_name == "RD":
                state1_raw = min((0, 1), key=lambda r: raw_const[r])
            else:
                raise ValueError(f"Unknown measure: {measure_name}")

            state2_raw = 1 - state1_raw

            # B) Intercept / conditional-level equality
            i1 = const_idx[state1_raw]
            i2 = const_idx[state2_raw]

            R_const = np.zeros((1, len(params)))
            R_const[0, i1] = 1.0
            R_const[0, i2] = -1.0

            wald, p_value, restriction, df = _wald_linear_restriction(params, cov, R_const)
            diff = float(restriction[0]) if np.all(np.isfinite(restriction)) else np.nan
            var_diff = float((R_const @ cov @ R_const.T).item())
            se_diff = np.sqrt(var_diff) if var_diff > 0 else np.nan

            individual_rows.append({"shock": uncertainty_name, "measure": measure_name,
                "market": market, "test_type": "intercept", "variable": "const",
                "state1_raw_regime": state1_raw, "state2_raw_regime": state2_raw,
                "estimate_state1": params[i1], "estimate_state2": params[i2],
                "difference_state1_minus_state2": diff, "se_difference": se_diff, "wald_chi2": wald,
                "df": df, "p_value": p_value})

            # C) Variable-specific slope equality tests
            slope_restrictions = []

            for variable in exog_cols:
                j1 = _find_regime_parameter_index(names, variable, state1_raw)
                j2 = _find_regime_parameter_index(names, variable, state2_raw)

                if j1 is None or j2 is None:
                    raise ValueError(f"Cannot find {variable} coefficients for "
                        f"{uncertainty_name}-{measure_name}-{market}.")

                R_var = np.zeros((1, len(params)))
                R_var[0, j1] = 1.0
                R_var[0, j2] = -1.0
                slope_restrictions.append(R_var[0])

                wald, p_value, restriction, df = _wald_linear_restriction(params, cov, R_var)
                diff = (float(restriction[0])
                    if np.all(np.isfinite(restriction))
                    else np.nan)
                var_diff = float((R_var @ cov @ R_var.T).item())
                se_diff = np.sqrt(var_diff) if var_diff > 0 else np.nan

                beta1 = float(params[j1])
                beta2 = float(params[j2])

                individual_rows.append({"shock": uncertainty_name, "measure": measure_name,
                    "market": market, "test_type": "slope", "variable": variable,
                    "state1_raw_regime": state1_raw, "state2_raw_regime": state2_raw,
                    "estimate_state1": beta1, "estimate_state2": beta2,
                    "difference_state1_minus_state2": diff, "se_difference": se_diff,
                    "sign_opposite": (beta1 * beta2) < 0, "wald_chi2": wald, "df": df,
                    "p_value": p_value})

            # D) Joint equality of all four macrofinancial slopes
            R_joint = np.vstack(slope_restrictions)
            wald_joint, p_joint, _, df_joint = _wald_linear_restriction(params, cov, R_joint)

            joint_rows.append({"shock": uncertainty_name, "measure": measure_name,
                "market": market, "test": "all four slopes equal across states",
                "wald_chi2": wald_joint, "df": df_joint, "p_value": p_joint})

individual_tests = pd.DataFrame(individual_rows)
joint_tests = pd.DataFrame(joint_rows)

# E) Benjamini-Hochberg correction in three independent families:
# 30 conditional levels; 120 individual slopes; 30 joint-slope tests (ALL).
# The existing individual-slope family and its row selection stay unchanged.
individual_tests["q_value_bh"] = np.nan
for family in ("intercept", "slope"):
    valid = (individual_tests["test_type"] == family) & individual_tests["p_value"].notna()
    if valid.any():
        individual_tests.loc[valid, "q_value_bh"] = multipletests(
            individual_tests.loc[valid, "p_value"].to_numpy(), alpha=0.05, method="fdr_bh")[1]
joint_tests["q_value_bh"] = np.nan
valid_joint = joint_tests["p_value"].notna()
if valid_joint.any():
    joint_tests.loc[valid_joint, "q_value_bh"] = multipletests(
        joint_tests.loc[valid_joint, "p_value"].to_numpy(), alpha=0.05, method="fdr_bh")[1]

individual_tests["stars_p"] = individual_tests["p_value"].apply(_stars)
individual_tests["stars_q"] = individual_tests["q_value_bh"].apply(_stars)
joint_tests["stars_p"] = joint_tests["p_value"].apply(_stars)
joint_tests["stars_q"] = joint_tests["q_value_bh"].apply(_stars)

# More readable ordering
individual_tests = individual_tests.sort_values(
    ["shock", "measure", "market", "test_type", "variable"]).reset_index(drop=True)
joint_tests = joint_tests.sort_values(["shock", "measure", "market"]).reset_index(drop=True)

# F) Compact diagnostic summaries
intercept_tests = individual_tests[individual_tests["test_type"] == "intercept"].copy()
slope_tests = individual_tests[individual_tests["test_type"] == "slope"].copy()

# G) Export
if EXPORT_COMPREHENSIVE_TESTS_XLSX:
    _test_output_dir.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(COMPREHENSIVE_TEST_OUTPUT_PATH, engine="openpyxl") as writer:
        individual_tests.to_excel(writer, sheet_name="individual_tests", index=False)
        joint_tests.to_excel(writer, sheet_name="joint_slope_tests", index=False)

        for sheet_name in ["individual_tests", "joint_slope_tests"]:
            ws = writer.sheets[sheet_name]
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions

        format_workbook(writer)


## 5. Supplementary conditional diagnostics

Reuse the fitted states and full-sample standardization. No additional Markov model is fitted.


In [5]:
HAC_LAGS = 6


PROBABILITY_FLOOR = 1e-6


PRIMARY_DATE = "2021-01-01"


INDICATOR_DATES = ("2020-03-01", PRIMARY_DATE, "2022-02-01")


EXCLUDED_START, EXCLUDED_END = "2020-03-01", "2020-12-01"


EXOG_COLUMNS = ("DXY", "GSCPI", "CPI", "Industrial")


def _hac_fit(y, design, weights=None, maxlags=HAC_LAGS):
    model = sm.OLS(y, design) if weights is None else sm.WLS(y, design, weights=weights)
    return model.fit(cov_type="HAC", cov_kwds={"maxlags": maxlags,
                                             "use_correction": False}, use_t=False)


def _state_probability(result, raw_state):
    probabilities = result.smoothed_marginal_probabilities
    if isinstance(probabilities, pd.DataFrame):
        values = probabilities.iloc[:, raw_state].to_numpy(dtype=float)
    else:
        values = np.asarray(probabilities, dtype=float)
        if values.shape[0] != int(result.nobs):
            values = values.T
        values = values[:, raw_state]
    if not np.isfinite(values).all() or np.any((values < -1e-10) | (values > 1 + 1e-10)):
        raise ValueError("Supplementary diagnostics require valid smoothed probabilities.")
    # Preserve the 1e-6 numerical floor in the scripts used to construct the
    # appendix. This copy is not written back to the probability workbooks.
    return np.clip(values, PROBABILITY_FLOOR, 1 - PROBABILITY_FLOOR)


def run_appendix_diagnostics(results, probability_tables, state_mapping_tables,
                            conditioning_data, coefficients, individual_tests):
    """Return the three supplementary diagnostic tables and full numeric detail."""
    conditioning = conditioning_data[["date", *EXOG_COLUMNS]].copy()
    conditioning["date"] = pd.to_datetime(conditioning["date"], utc=True).dt.tz_convert(None)
    if conditioning["date"].duplicated().any():
        raise ValueError("Conditioning dates must be unique for the diagnostic date merge.")
    selected = individual_tests.loc[(individual_tests.test_type == "slope") &
                                    (individual_tests.q_value_bh < .05)].copy()
    fitted, exposure_rows, linear_rows, sample_rows = {}, [], [], []
    for shock, measures in results.items():
        mappings = state_mapping_tables[shock]
        for measure, markets in measures.items():
            for market, result in markets.items():
                key = {"shock": shock, "measure": measure, "market": market}
                mask = (probability_tables[shock].measure == measure) & (probability_tables[shock].market == market)
                sample = probability_tables[shock].loc[mask, ["date", "metric_value"]].copy()
                sample["date"] = pd.to_datetime(sample["date"], utc=True).dt.tz_convert(None)
                sample = sample.merge(conditioning, on="date", how="left", validate="one_to_one").sort_values("date")
                if len(sample) != int(result.nobs) or sample.isna().any().any():
                    raise ValueError(f"Diagnostic observations do not match the fitted model: {key}")
                y = sample.metric_value.to_numpy(dtype=float)
                x = sample[list(EXOG_COLUMNS)].to_numpy(dtype=float)
                design = np.column_stack([np.ones(len(y)), x])
                if not np.allclose(np.asarray(result.model.exog), design, rtol=0, atol=1e-12):
                    raise ValueError(f"Diagnostic conditioning vectors differ from the original fit: {key}")
                if not np.allclose(np.asarray(result.model.endog), y - y.mean(), rtol=0, atol=1e-12):
                    raise ValueError(f"Diagnostic outcomes differ from the original fit: {key}")
                mapping = mappings.loc[(mappings.measure == measure) & (mappings.market == market)].iloc[0]
                p1 = _state_probability(result, int(mapping.state1_raw))
                dates = pd.DatetimeIndex(sample.date)
                pre = dates < PRIMARY_DATE
                keep = ~((dates >= EXCLUDED_START) & (dates <= EXCLUDED_END))
                local = coefficients.loc[(coefficients.shock == shock) &
                                         (coefficients.measure == measure) &
                                         (coefficients.market == market)]
                beta = {state: local.loc[local.state == state].set_index("term").loc[
                    list(EXOG_COLUMNS), "estimate"].to_numpy(dtype=float) for state in (1, 2)}
                linear = _hac_fit(y, design)
                for i, variable in enumerate(EXOG_COLUMNS, 1):
                    linear_rows.append({**key, "variable": variable, "estimate": linear.params[i],
                                        "hac_t": linear.tvalues[i], "p_value": linear.pvalues[i]})
                exposure_rows.append({**key,
                    **{f"sd_state{state}": float(np.std(x @ beta[state], ddof=0)) for state in (1, 2)},
                    **{f"sd_excl2020_state{state}": float(np.std(x[keep] @ beta[state], ddof=0)) for state in (1, 2)}})
                sample_rows.append({**key, "n_full": len(y), "n_pre2021": int(pre.sum()),
                                    "n_excl2020": int(keep.sum()),
                                    **{f"n_indicator_{date[:7]}": int((dates >= date).sum()) for date in INDICATOR_DATES}})
                fitted[(shock, measure, market)] = dict(y=y, x=x, design=design, dates=dates,
                    p1=p1, beta=beta, linear=linear, pre=pre, keep=keep)

    exposure = pd.DataFrame(exposure_rows)
    linear_table = pd.DataFrame(linear_rows)
    slope_coefficients = coefficients.loc[coefficients.term.isin(EXOG_COLUMNS)]
    counts = slope_coefficients.loc[slope_coefficients.p_value < .05].groupby("state").size()
    linear_counts = linear_table.loc[linear_table.p_value < .05].groupby("variable").size()
    macro_summary = pd.DataFrame([
        {"diagnostic": "greater_dispersion_full", "value": int((exposure.sd_state2 > exposure.sd_state1).sum())},
        {"diagnostic": "greater_dispersion_excl2020", "value": int((exposure.sd_excl2020_state2 > exposure.sd_excl2020_state1).sum())},
        *[{"diagnostic": f"within_state{state}_detections", "value": int(counts.get(state, 0))} for state in (1, 2)],
        *[{"diagnostic": f"linear_{variable}_detections", "value": int(linear_counts.get(variable, 0))} for variable in EXOG_COLUMNS],
    ])

    subsample_rows, period_rows, detail_rows, alignment_rows = [], [], [], []
    for _, test in selected.iterrows():
        key = {name: test[name] for name in ("shock", "measure", "market", "variable")}
        data = fitted[(test.shock, test.measure, test.market)]
        position = EXOG_COLUMNS.index(test.variable) + 1
        ms = {state: data["beta"][state][position - 1] for state in (1, 2)}
        baseline_gap = ms[2] - ms[1]
        if not np.isfinite(baseline_gap) or baseline_gap == 0:
            raise ValueError(f"A selected slope difference has an undefined ratio: {key}")
        row = {**key, "linear": data["linear"].params[position],
               "linear_p": data["linear"].pvalues[position], "ms_s1": ms[1], "ms_s2": ms[2]}
        for label, mask in (("full", np.ones(len(data["y"]), dtype=bool)),
                            ("pre2021", data["pre"]), ("excl2020", data["keep"])):
            # Retained observations become adjacent for HAC after the 2020 rows
            # are removed, exactly as stated in the supplementary table notes.
            for state, weights in ((1, data["p1"]), (2, 1 - data["p1"])):
                fit = _hac_fit(data["y"][mask], data["design"][mask], weights[mask])
                row[f"{label}_s{state}"] = fit.params[position]
                row[f"{label}_s{state}_t"] = fit.tvalues[position]
                if label == "full":
                    alignment_rows.append({**key, "state": state, "markov_slope": ms[state],
                        "weighted_slope": fit.params[position],
                        "absolute_difference": abs(fit.params[position] - ms[state])})
        subsample_rows.append(row)

        period = {**key, "ms_s1": ms[1], "ms_s2": ms[2]}
        for date in INDICATOR_DATES:
            indicator = (data["dates"] >= date).astype(float)
            design = np.column_stack([data["design"], indicator])
            fits = {state: _hac_fit(data["y"], design, weights) for state, weights in (
                (1, data["p1"]), (2, 1 - data["p1"]))}
            gap = fits[2].params[position] - fits[1].params[position]
            ratio = gap / baseline_gap
            tag = date[:7].replace("-", "")
            period[f"ratio_{tag}"] = ratio
            detail = {**key, "indicator_start": date, "nobs": len(indicator),
                      "n_indicator": int(indicator.sum()), "difference_ms_s2_minus_s1": baseline_gap,
                      "difference_indicator_s2_minus_s1": gap, "signed_ratio": ratio}
            for state, fit in fits.items():
                detail.update({f"slope_s{state}": fit.params[position], f"slope_s{state}_t": fit.tvalues[position],
                               f"indicator_s{state}": fit.params[-1], f"indicator_s{state}_t": fit.tvalues[-1]})
                if date == PRIMARY_DATE:
                    period.update({f"withD_s{state}": fit.params[position], f"withD_s{state}_t": fit.tvalues[position],
                                   f"indicator_s{state}": fit.params[-1], f"indicator_s{state}_t": fit.tvalues[-1]})
            detail_rows.append(detail)
        period_rows.append(period)

    parameters = pd.DataFrame([
        {"setting": "HAC lags", "value": HAC_LAGS},
        {"setting": "HAC kernel", "value": "Bartlett"},
        {"setting": "HAC small-sample correction", "value": False},
        {"setting": "HAC normal reference (use_t)", "value": False},
        {"setting": "Probability floor", "value": PROBABILITY_FLOOR},
        {"setting": "Primary indicator starts", "value": PRIMARY_DATE},
        {"setting": "Alternative indicator dates", "value": ", ".join(d for d in INDICATOR_DATES if d != PRIMARY_DATE)},
        {"setting": "Standardization", "value": "Existing full-sample standardized X; no subset rescaling"},
        {"setting": "Latent states", "value": "Existing full-sample smoothed probabilities; no state refit"},
        {"setting": "Selection", "value": "Original individual-slope BH q < 0.05; no new BH adjustment"},
        {"setting": "Ratio", "value": "(beta_2,D - beta_1,D) / (beta_2,MS - beta_1,MS); signed"},
    ])
    return dict(macro_summary=macro_summary, fitted_dispersion=exposure, linear_hac=linear_table,
                subsample_checks=pd.DataFrame(subsample_rows), period_indicator=pd.DataFrame(period_rows),
                indicator_dates=pd.DataFrame(detail_rows), weighted_alignment=pd.DataFrame(alignment_rows),
                diagnostic_samples=pd.DataFrame(sample_rows), diagnostic_settings=parameters)


# Reuse the fitted models; do not estimate another Markov process.
appendix_diagnostics = run_appendix_diagnostics(
    results, probability_tables, state_mapping_tables, x_data,
    coefficient_table, individual_tests)


## 6. Final tables and report


In [6]:
# Optional manuscript verification. Normal analysis needs no manuscript files.
VERIFY_MANUSCRIPT = False
APPENDIX_TEX = None  # Set a path to Supplementary_Appendix.tex when enabled.
MAIN_TEX = None      # Optional path to main_J1108.tex, including Panel C checks.

# One final, formatted output. Expand coefficient tables when needed.
final_input_hashes = {p: hashlib.sha256((PROJECT_ROOT / p).read_bytes()).hexdigest()
                     for p in input_sha256}
if final_input_hashes != input_sha256:
    raise RuntimeError("An input workbook or CSV changed during this run.")

report_html = write_markov_report(
    OUTPUT_DIR, paper_tables, model_overview, coefficient_table,
    state_mapping_tables, individual_tests, joint_tests, model_warnings,
    exported_probability_files, input_sha256, appendix_diagnostics=appendix_diagnostics)
if VERIFY_MANUSCRIPT:
    if APPENDIX_TEX is None:
        raise ValueError("Set APPENDIX_TEX before enabling manuscript verification.")
    manuscript_comparisons, manuscript_verification = verify_manuscript_files(
        APPENDIX_TEX, OUTPUT_DIR / "Markov_results.xlsx", MAIN_TEX)

display(HTML(report_html))


Market,State 1 mean,State 2 mean,p11,p22,"E(D1), months","E(D2), months"
OIL,0.8251,0.6975,0.9780,0.9427,45.4,17.4
NG,0.7768,0.6359,0.9847,0.9681,65.4,31.3
COAL,0.7562,0.7480,0.9502,0.9502,20.1,20.1
ERIX,0.8013,0.7122,0.9365,0.9164,15.8,12.0
EUA,0.8165,0.7134,0.9507,0.9529,20.3,21.2
Market,State,Conditional mean,DXY,GSCPI,CPI,Industrial
OIL,1,0.8251*** (313.762),-0.0031 (-1.247),0.0005 (0.194),-0.0038 (-1.100),-0.0006 (-0.272)
OIL,2,0.6975*** (57.447),0.0119 (1.290),0.0218** (2.323),-0.0143* (-1.684),0.0305 (0.990)
NG,1,0.7768*** (177.820),-0.0063 (-1.510),-0.0043 (-0.914),-0.0013 (-0.302),0.0001 (0.022)
NG,2,0.6359*** (67.474),0.0020 (0.198),-0.0111 (-1.264),-0.0253** (-2.484),-0.0107 (-1.170)
